In [ ]:
#| default_exp langgraph_viz


# LangGraph + Plotly viz

Deterministic LangGraph orchestrator: resolve intent → fetch model data (mat-gram) → build a Plotly figure. One HTTP service on `:8002/viz` replaces the old per-chart microservices.

Run the server with `pixi run serve-viz` (also starts settings on `:8011`).


In [ ]:
#| export
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Callable, TypedDict

import plotly.graph_objects as go
import uvicorn
from langgraph.graph import END, START, StateGraph
from starlette.applications import Starlette
from starlette.middleware.cors import CORSMiddleware
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route

REPO = Path(__file__).resolve().parents[1]
SERVICES = REPO / "services"
for p in (REPO, SERVICES):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from matgram_client import MatgramError  # noqa: E402
from viz.plotly_viz import (  # noqa: E402
    build_attention_heatmap_figure,
    build_encode_pass_figure,
    build_latent_interpolation_figure,
    build_molecular_structure_figure,
    build_parity_figure,
    build_roundtrip_figure,
)

PORT = 8002
PATH = "/viz"


## State + chart tools


In [ ]:
#| export
class VizState(TypedDict, total=False):
    message: str | None
    endpoint: str
    visualization: str
    body: dict[str, Any]
    figure_payload: dict[str, Any] | None
    error: str | None
    meta: dict[str, Any]
    status_code: int

Builder = Callable[..., go.Figure]


def _placeholder_scatter(*, message: str | None = None, body: dict[str, Any] | None = None, **_: Any) -> go.Figure:
    fig = go.Figure(
        data=[go.Scatter(x=[20, 40, 60], y=[12.4, 28.1, 51.0], mode="lines+markers", marker={"size": 10})],
        layout={"title": "Solubility scatter", "xaxis": {"title": "temperature_C"}, "yaxis": {"title": "solubility_g_per_L"}},
    )
    fig._matgram_meta = {"placeholder": True, "source": "demo"}  # type: ignore[attr-defined]
    return fig


def _placeholder_heatmap(*, message: str | None = None, body: dict[str, Any] | None = None, **_: Any) -> go.Figure:
    fig = go.Figure(
        data=[go.Heatmap(
            x=["water", "ethanol", "acetone"], y=["20", "40", "60"],
            z=[[12.4, 8.1, 5.0], [28.1, 18.4, 11.2], [51.0, 33.7, 19.5]], colorscale="YlGnBu",
        )],
        layout={"title": "Solubility heatmap", "xaxis": {"title": "solvent"}, "yaxis": {"title": "temperature_C"}},
    )
    fig._matgram_meta = {"placeholder": True, "source": "demo"}  # type: ignore[attr-defined]
    return fig


def _placeholder_timeseries(*, message: str | None = None, body: dict[str, Any] | None = None, **_: Any) -> go.Figure:
    fig = go.Figure(
        data=[go.Scatter(
            x=[0, 15, 30, 60], y=[10.0, 18.2, 27.5, 41.0],
            mode="lines+markers", marker={"size": 9}, line={"width": 3},
        )],
        layout={"title": "Solubility timeseries", "xaxis": {"title": "time_min"}, "yaxis": {"title": "solubility_g_per_L"}},
    )
    fig._matgram_meta = {"placeholder": True, "source": "demo"}  # type: ignore[attr-defined]
    return fig


VIZ_BUILDERS: dict[str, tuple[str, Builder]] = {
    "scatter": ("solubility_scatter", _placeholder_scatter),
    "heatmap": ("solubility_heatmap", _placeholder_heatmap),
    "timeseries": ("solubility_timeseries", _placeholder_timeseries),
    "encode_pass": ("solubility_encode_pass", build_encode_pass_figure),
    "attention_heatmap": ("solubility_attention_heatmap", build_attention_heatmap_figure),
    "property_diagnostics": ("solubility_property_diagnostics", build_parity_figure),
    "parity": ("solubility_property_diagnostics", build_parity_figure),
    "latent_interpolation": ("solubility_latent_interpolation", build_latent_interpolation_figure),
    "molecular_structure": ("solubility_molecular_structure", build_molecular_structure_figure),
    "roundtrip": ("solubility_roundtrip", build_roundtrip_figure),
}

KEYWORD_TO_VIZ: list[tuple[str, str]] = [
    ("encode_pass", "encode_pass"), ("encode-pass", "encode_pass"),
    ("t-sne", "encode_pass"), ("tsne", "encode_pass"), ("embedding", "encode_pass"),
    ("attention", "attention_heatmap"), ("similarity", "attention_heatmap"),
    ("parity", "property_diagnostics"), ("property_diagnostics", "property_diagnostics"),
    ("latent", "latent_interpolation"), ("interpolat", "latent_interpolation"),
    ("molecular_structure", "molecular_structure"), ("molecule", "molecular_structure"),
    ("roundtrip", "roundtrip"), ("round-trip", "roundtrip"),
    ("timeseries", "timeseries"), ("time-series", "timeseries"),
    ("heatmap", "heatmap"), ("scatter", "scatter"),
]


def normalize_viz_id(raw: str | None) -> str | None:
    if not raw:
        return None
    s = str(raw).strip()
    if not s:
        return None
    for prefix in ("solubility_", "settings_"):
        if s.startswith(prefix):
            s = s[len(prefix):]
            break
    s = s.replace("-", "_")
    if s == "parity":
        return "property_diagnostics"
    return s if s in VIZ_BUILDERS else None


def resolve_visualization(
    *,
    endpoint: str | None = None,
    visualization: str | None = None,
    message: str | None = None,
) -> str:
    """Pick a visualization id from explicit fields or message keywords."""
    for candidate in (visualization, endpoint):
        vid = normalize_viz_id(candidate)
        if vid and vid in VIZ_BUILDERS:
            return vid
    text = (message or "").lower()
    for needle, vid in KEYWORD_TO_VIZ:
        if needle in text and vid in VIZ_BUILDERS:
            return vid
    for vid in VIZ_BUILDERS:
        if vid in text or f"solubility_{vid}" in text:
            return vid
    return "encode_pass"


def figure_to_payload(fig: Any) -> dict[str, Any]:
    payload = json.loads(fig.to_json())
    return {"data": payload.get("data", []), "layout": payload.get("layout", {})}


def build_viz_figure(
    visualization: str,
    *,
    message: str | None,
    body: dict[str, Any] | None,
) -> tuple[go.Figure, str, dict[str, Any]]:
    """Run the chart builder for ``visualization``. Returns (fig, endpoint, meta)."""
    if visualization not in VIZ_BUILDERS:
        raise ValueError(f"Unknown visualization: {visualization}")
    endpoint, builder = VIZ_BUILDERS[visualization]
    fig = builder(message=message, body=body or {})
    meta = dict(getattr(fig, "_matgram_meta", None) or {})
    return fig, endpoint, meta


## LangGraph


In [ ]:
#| export
def resolve_intent(state: VizState) -> dict[str, Any]:
    body = state.get("body") or {}
    visualization = resolve_visualization(
        endpoint=state.get("endpoint") or body.get("endpoint"),
        visualization=state.get("visualization") or body.get("visualization"),
        message=state.get("message") or body.get("message"),
    )
    return {
        "visualization": visualization,
        "endpoint": f"solubility_{visualization}",
        "error": None,
        "status_code": 200,
    }


def run_plan(state: VizState) -> dict[str, Any]:
    visualization = state.get("visualization") or "encode_pass"
    message = state.get("message")
    body = state.get("body") or {}
    try:
        fig, endpoint, meta = build_viz_figure(visualization, message=message, body=body)
        return {
            "endpoint": endpoint,
            "figure_payload": figure_to_payload(fig),
            "meta": meta,
            "error": None,
            "status_code": 200,
        }
    except MatgramError as exc:
        return {"figure_payload": None, "meta": {}, "error": str(exc), "status_code": 503}
    except Exception as exc:  # noqa: BLE001
        return {"figure_payload": None, "meta": {}, "error": str(exc), "status_code": 500}


def build_graph():
    g = StateGraph(VizState)
    g.add_node("resolve_intent", resolve_intent)
    g.add_node("run_plan", run_plan)
    g.add_edge(START, "resolve_intent")
    g.add_edge("resolve_intent", "run_plan")
    g.add_edge("run_plan", END)
    return g.compile()


_GRAPH = None


def get_graph():
    global _GRAPH
    if _GRAPH is None:
        _GRAPH = build_graph()
    return _GRAPH


def run_viz(
    message: str | None = None,
    *,
    endpoint: str | None = None,
    visualization: str | None = None,
    body: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Run the viz graph and return a UI-compatible response dict."""
    body = dict(body or {})
    if message is not None and "message" not in body:
        body["message"] = message
    if endpoint and "endpoint" not in body:
        body["endpoint"] = endpoint
    if visualization and "visualization" not in body:
        body["visualization"] = visualization

    result = get_graph().invoke(
        {
            "message": message if message is not None else body.get("message"),
            "endpoint": endpoint or body.get("endpoint") or "",
            "visualization": visualization or body.get("visualization") or "",
            "body": body,
            "figure_payload": None,
            "error": None,
            "meta": {},
            "status_code": 200,
        }
    )

    ep = result.get("endpoint") or endpoint or "solubility_viz"
    viz = result.get("visualization") or visualization or "viz"
    meta = result.get("meta") or {}
    err = result.get("error")
    status_code = int(result.get("status_code") or 200)

    if err or not result.get("figure_payload"):
        out: dict[str, Any] = {
            "status": "error",
            "endpoint": ep,
            "visualization": viz,
            "message": body.get("message"),
            "error": err or "No plotly figure produced",
            "status_code": status_code if status_code >= 400 else 500,
        }
        if status_code == 503:
            out["hint"] = (
                "Start mat-gram locally (`pixi run api`) and POST /load, "
                "then set MATGRAM_API_URL if needed."
            )
        return out

    return {
        "status": "ok",
        "endpoint": ep,
        "visualization": viz,
        "message": body.get("message"),
        "placeholder": bool(meta.get("placeholder", False)),
        "source": meta.get("source", "matgram"),
        "model": meta.get("model"),
        "mode": meta.get("mode"),
        "task": meta.get("task"),
        "plotly": result["figure_payload"],
        "status_code": 200,
    }


## HTTP service


In [ ]:
#| export
def make_app(routes: list[Route]) -> Starlette:
    app = Starlette(routes=routes)
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],
        allow_methods=["*"],
        allow_headers=["*"],
    )
    return app


async def viz_handler(request: Request) -> JSONResponse:
    body = (
        await request.json()
        if request.headers.get("content-type", "").startswith("application/json")
        else {}
    )
    if not isinstance(body, dict):
        body = {}
    endpoint = body.get("endpoint") or request.query_params.get("endpoint") or ""
    visualization = body.get("visualization") or request.query_params.get("visualization") or ""
    result = run_viz(
        body.get("message"),
        endpoint=str(endpoint) if endpoint else None,
        visualization=str(visualization) if visualization else None,
        body=body,
    )
    status_code = int(result.pop("status_code", 200))
    return JSONResponse(result, status_code=status_code)


async def health_handler(_: Request) -> JSONResponse:
    return JSONResponse({"status": "ok", "service": "langgraph_viz"})


def create_viz_app() -> Starlette:
    return make_app(
        [
            Route(PATH, viz_handler, methods=["POST"]),
            Route("/health", health_handler, methods=["GET"]),
        ]
    )


def serve_viz(host: str = "127.0.0.1", port: int = PORT) -> None:
    """Serve the LangGraph viz app (blocking)."""
    print(f"Serving LangGraph viz on http://{host}:{port}{PATH}")
    uvicorn.run(create_viz_app(), host=host, port=port)


app = create_viz_app()


if __name__ == "__main__":
    serve_viz()


In [ ]:
#| hide
# Quick local check (placeholder — no mat-gram required)
out = run_viz("demo", endpoint="solubility_scatter")
assert out["status"] == "ok" and out["plotly"]["data"], out
out["endpoint"], len(out["plotly"]["data"])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
